In [1]:
!pip install tensorflow

In [2]:
# biblioteca
import pandas as pd
import tensorflow as tf
import sklearn

In [3]:
from tensorflow.keras.layers import Dense, Dropout, Activation, Input
from tensorflow.keras.models import Model
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [4]:
base = pd.read_csv('games.csv')
base

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16714,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,Tecmo Koei,0.00,0.00,0.01,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16715,LMA Manager 2007,X360,2006.0,Sports,Codemasters,0.00,0.01,0.00,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16716,Haitaka no Psychedelica,PSV,2016.0,Adventure,Idea Factory,0.00,0.00,0.01,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16717,Spirits & Spells,GBA,2003.0,Platform,Wanadoo,0.01,0.00,0.00,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# apagando alguns campos
base = base.drop('Other_Sales', axis = 1)
base = base.drop('Global_Sales', axis = 1)
base = base.drop('Developer', axis = 1)

In [6]:
base.shape

(16719, 13)

In [7]:
base.isnull().sum() # somatorio dos valores null de cada coluna

,0
Name,2
Platform,0
Year_of_Release,269
Genre,2
Publisher,54
NA_Sales,0
EU_Sales,0
JP_Sales,0
Critic_Score,8582
Critic_Count,8582


In [8]:
base = base.dropna(axis = 0)

In [9]:
base.shape

(6825, 13)

In [10]:
base.isnull().sum() # somatorio dos valores null de cada coluna

,0
Name,0
Platform,0
Year_of_Release,0
Genre,0
Publisher,0
NA_Sales,0
EU_Sales,0
JP_Sales,0
Critic_Score,0
Critic_Count,0


In [11]:
base['Name'].value_counts() # contagem dos nomes diferentes

,count
Name,
Madden NFL 07,8
Need for Speed: Most Wanted,8
LEGO Star Wars II: The Original Trilogy,8
Terraria,7
Madden NFL 08,7
...,...
Brain Age: Train Your Brain in Minutes a Day,1
Wii Fit,1
Wii Sports Resort,1


In [12]:
base = base.drop('Name', axis = 1) # apagar os nomes

In [15]:
base.columns # para vizualizar a ordem

Index(['Platform', 'Year_of_Release', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Critic_Score', 'Critic_Count', 'User_Score',
       'User_Count', 'Rating'],
      dtype='object')

In [18]:
X = base.iloc[:, [0,1,2,3,7,8,9,10,11]].values
X # todos os atributos previsores

array([['Wii', 2006.0, 'Sports', ..., '8', 322.0, 'E'],
       ['Wii', 2008.0, 'Racing', ..., '8.3', 709.0, 'E'],
       ['Wii', 2009.0, 'Sports', ..., '8', 192.0, 'E'],
       ...,
       ['PC', 2014.0, 'Action', ..., '7.6', 412.0, 'M'],
       ['PC', 2011.0, 'Shooter', ..., '5.8', 43.0, 'T'],
       ['PC', 2011.0, 'Strategy', ..., '7.2', 13.0, 'E10+']], dtype=object)

In [19]:
y_na = base.iloc[:, 4].values
y_eu = base.iloc[:, 5].values
y_jp = base.iloc[:, 6].values
# variaveis dos locais na, eu, jp

In [20]:
y_na

array([4.136e+01, 1.568e+01, 1.561e+01, ..., 0.000e+00, 1.000e-02,
       0.000e+00])

In [21]:
y_eu

array([2.896e+01, 1.276e+01, 1.093e+01, ..., 1.000e-02, 0.000e+00,
       1.000e-02])

In [22]:
y_jp

array([3.77, 3.79, 3.28, ..., 0.  , 0.  , 0.  ])

In [24]:
# PS2 1 0 0 0 0 ...
# X360 0 1 0 0 ...
base['Platform'].value_counts() # contagem de plataformas

,count
Platform,
PS2,1140
X360,858
PS3,769
PC,651
XB,565
Wii,479
DS,464
PSP,390
GC,348


In [25]:
base.columns

Index(['Platform', 'Year_of_Release', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Critic_Score', 'Critic_Count', 'User_Score',
       'User_Count', 'Rating'],
      dtype='object')

In [26]:
from numpy import remainder
onehotencoder = ColumnTransformer(transformers = [("OneHot", OneHotEncoder(), [0, 2, 3, 8])], remainder = 'passthrough')
X = onehotencoder.fit_transform(X).toarray()

In [29]:
X.shape

(6825, 303)

In [30]:
X[0]

array([0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 

In [31]:
(303 + 3) / 2

153.0

In [32]:
camada_entrada = Input(shape = (303,))
camada_oculta1 = Dense(units = 153, activation='relu')(camada_entrada)
camada_oculta2 = Dense(units = 153, activation='relu')(camada_oculta1)
camada_saida1 = Dense(units = 1, activation='linear')(camada_oculta2)
camada_saida2 = Dense(units = 1, activation='linear')(camada_oculta2)
camada_saida3 = Dense(units = 1, activation='linear')(camada_oculta2)

In [33]:
regressor = Model(inputs = camada_entrada, outputs = [camada_saida1, camada_saida2, camada_saida3])

In [34]:
regressor.compile(optimizer='adam', loss = 'mse')

In [35]:
regressor.fit(X, [y_na, y_eu, y_jp], epochs=500, batch_size=100)

Epoch 1/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - dense_2_loss: 258.1945 - dense_3_loss: 1168.4600 - dense_4_loss: 52.7235 - loss: 1495.5490 
Epoch 2/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - dense_2_loss: 6.2123 - dense_3_loss: 4.1818 - dense_4_loss: 3.4583 - loss: 13.6947
Epoch 3/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - dense_2_loss: 3.4310 - dense_3_loss: 3.1846 - dense_4_loss: 4.3571 - loss: 11.0776
Epoch 4/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - dense_2_loss: 1.6862 - dense_3_loss: 1.3479 - dense_4_loss: 0.7127 - loss: 3.7740
Epoch 5/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - dense_2_loss: 2.3280 - dense_3_loss: 1.4180 - dense_4_loss: 2.2306 - loss: 6.0281
Epoch 6/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - dense_2_loss: 2.4553 - dense_3_loss: 2.4834 - dense_4_loss: 0.5413 - loss: 5.3797
Epoch 7/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - dense_2_loss: 2.8964 - dense_3_loss: 2.1327 - dense_4_loss: 5.4848 - loss: 10.6148
Epoch 8/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0

In [36]:
previsao_na, previsao_eu, previsao_jp = regressor.predict(X) # Testando o modelo com os próprios dados de treinamento

214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [37]:
previsao_na, previsao_na.mean() # previsao e media da previsao de na

(array([[15.514455  ],
        [11.859058  ],
        [12.649443  ],
        ...,
        [-0.2290887 ],
        [-0.12773108],
        [-0.12834409]], dtype=float32),
 np.float32(0.105369195))

In [38]:
y_na, y_na.mean() # valores de na, media da variavel de na

(array([4.136e+01, 1.568e+01, 1.561e+01, ..., 0.000e+00, 1.000e-02,
        0.000e+00]),
 np.float64(0.3944835164835165))

In [39]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_na, previsao_na) # erro medio absoluto

0.3170966082050687

In [40]:
previsao_eu, previsao_eu.mean() # previsao e media de eu

(array([[11.151625  ],
        [ 8.872805  ],
        [ 9.146378  ],
        ...,
        [ 0.01796886],
        [-0.04424992],
        [-0.02651668]], dtype=float32),
 np.float32(0.08532156))

In [42]:
y_eu, y_eu.mean() # valores de eu, media da variavel de eu

(array([2.896e+01, 1.276e+01, 1.093e+01, ..., 1.000e-02, 0.000e+00,
        1.000e-02]),
 np.float64(0.23608937728937732))

In [44]:
mean_absolute_error(y_eu, previsao_eu) # erro medio absoluto

0.188535821437137

In [45]:
previsao_jp, previsao_jp.mean() # previsao e media de jp

(array([[ 3.4030685 ],
        [ 3.8285487 ],
        [ 2.9532096 ],
        ...,
        [-0.07239521],
        [-0.03867896],
        [-0.02835295]], dtype=float32),
 np.float32(0.0072390637))

In [46]:
y_jp, y_jp.mean() # valores de jp, media da variavel de jp

(array([3.77, 3.79, 3.28, ..., 0.  , 0.  , 0.  ]),
 np.float64(0.06415824175824175))

In [48]:
mean_absolute_error(y_jp, previsao_jp) # erro medio absoluto

0.08210895480424056